# SIEM/EDR Synthetic Data Generation

This notebook generates synthetic security data that mimics what a typical **Security Information and Event Management (SIEM)** or **Endpoint Detection and Response (EDR)** system would see across a corporate network.

## Key Features

- **Correlated Events**: All log types are interconnected - host network sessions drive firewall, IDS, DNS, proxy, and netflow logs
- **Stateful Sessions**: Network sessions follow realistic state machine transitions
- **Multi-Entity Coverage**: Hosts, users, firewalls, IDS/IPS, DNS, VPN, proxies, endpoints, netflow, and IAM/authentication
- **Referential Integrity**: Foreign keys ensure logs only reference valid source events

## Generated Log Types

| Log Type | Description | Correlation |
|----------|-------------|-------------|
| Host | Network endpoints (workstations, servers, IoT) | Core entity |
| User | Users operating on hosts | Assigned to hosts |
| Network Session | Host network activities | Drives all network logs |
| Firewall Log | Allow/deny decisions | From sessions |
| IDS Alert | Intrusion detection alerts | From suspicious sessions |
| DNS Log | DNS queries | From sessions |
| Proxy Log | Web proxy logs | From HTTP/S sessions |
| VPN Log | VPN connection events | From remote users |
| Endpoint Log | Process/file/registry events | From hosts |
| Netflow Record | Router flow records | From sessions |
| Auth Log | Authentication events | From users |

## Setup and Imports

In [1]:
import rockfish as rf
import rockfish.actions as ra
from rockfish.actions.ent import (
    CategoricalParams,
    Column,
    ColumnCategoryType,
    ColumnType,
    DataSchema,
    Derivation,
    DerivationFunctionType,
    Domain,
    DomainType,
    Entity,
    EntityRelationship,
    EntityRelationshipType,
    GlobalTimestamp,
    IDParams,
    MapValuesParams,
    NormalDistParams,
    SampleFromColumnParams,
    SequentialIntParams,
    StateMachineParams,
    Timestamp,
    Transition,
    UniformDistParams,
    ExponentialDistParams,
    TimeseriesParams,
)
from dotenv import load_dotenv
import pandas as pd
import numpy as np

In [2]:
# Connect to the Rockfish platform
load_dotenv()
conn = rf.Connection.from_env()

## Configuration Parameters

In [3]:
# Entity cardinalities
N_HOSTS = 200              # Network endpoints
N_USERS = 150              # System users
N_SESSIONS = 5000          # Network sessions (drives other logs)
N_VPN_EVENTS = 500         # VPN connection events
N_ENDPOINT_EVENTS = 10000  # EDR endpoint events
N_AUTH_EVENTS = 3000       # Authentication events

# Derived cardinalities (estimated based on session characteristics)
# - Firewall logs: ~1.5x sessions (multiple packets per session)
# - IDS alerts: ~5% of sessions
# - DNS logs: ~60% of sessions (external destinations need DNS)
# - Proxy logs: ~40% of sessions (HTTP/HTTPS traffic)
# - Netflow records: ~1x sessions

## Schema Design

### Entity Hierarchy

```
User (150)
  │
  ├──< Host (200) [user assigned to host]
  │      │
  │      ├──< NetworkSession (5000) [host initiates sessions]
  │      │      │
  │      │      ├──< FirewallLog [session generates FW logs]
  │      │      ├──< IDSAlert [session triggers alerts]
  │      │      ├──< DNSLog [session includes DNS queries]
  │      │      ├──< ProxyLog [HTTP sessions generate proxy logs]
  │      │      └──< NetflowRecord [session creates flow records]
  │      │
  │      └──< EndpointLog (10000) [host generates endpoint events]
  │
  ├──< VPNLog (500) [user connects via VPN]
  │
  └──< AuthLog (3000) [user authentication events]
```

## Create Data Schema

In [4]:
def create_siem_edr_schema(
    n_hosts: int = 200,
    n_users: int = 150,
    n_sessions: int = 5000,
    n_vpn_events: int = 500,
    n_endpoint_events: int = 10000,
    n_auth_events: int = 3000,
) -> DataSchema:
    """Create the SIEM/EDR data schema with correlated security events."""
    
    # ==========================================================================
    # ENTITY 1: user
    # Users that operate on hosts and authenticate to systems
    # ==========================================================================
    user = Entity(
        name="user",
        cardinality=n_users,
        columns=[
            Column(
                name="user_id",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.ID,
                    params=IDParams(template_str="USER_{id}"),
                ),
            ),
            Column(
                name="username",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.ID,
                    params=IDParams(template_str="user{id}"),
                ),
            ),
            Column(
                name="department",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=["Engineering", "Engineering", "Finance", "HR", "IT", "IT", "Sales", "Executive"],
                        with_replacement=True,
                    ),
                ),
            ),
            Column(
                name="role",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=["employee", "employee", "employee", "employee", "contractor", "admin", "service_account"],
                        with_replacement=True,
                    ),
                ),
            ),
            Column(
                name="is_privileged",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=["false", "false", "false", "false", "false", "true"],  # ~17% privileged
                        with_replacement=True,
                    ),
                ),
            ),
        ],
    )
    
    # ==========================================================================
    # ENTITY 2: host
    # Network endpoints - workstations, servers, IoT devices
    # ==========================================================================
    host = Entity(
        name="host",
        cardinality=n_hosts,
        columns=[
            Column(
                name="host_id",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.ID,
                    params=IDParams(template_str="HOST_{id}"),
                ),
            ),
            Column(
                name="hostname",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.ID,
                    params=IDParams(template_str="host{id}"),
                ),
            ),
            # Network segment determines IP range
            Column(
                name="segment",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=["segment_a", "segment_a", "segment_b", "segment_b", "segment_c", "dmz"],
                        with_replacement=True,
                    ),
                ),
            ),
            # IP address octets
            Column(
                name="ip_octet_1",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=[10, 10, 10, 172],  # Mostly 10.x.x.x, some DMZ 172.16.x.x
                        with_replacement=True,
                    ),
                ),
            ),
            Column(
                name="ip_octet_2",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=[1, 2, 3, 16],  # Segment IPs: 10.1.x.x, 10.2.x.x, 10.3.x.x, 172.16.x.x
                        with_replacement=True,
                    ),
                ),
            ),
            Column(
                name="ip_octet_3",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.UNIFORM_DIST,
                    params=UniformDistParams(lower=0, upper=255),
                ),
            ),
            Column(
                name="ip_octet_4",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.UNIFORM_DIST,
                    params=UniformDistParams(lower=1, upper=254),
                ),
            ),
            Column(
                name="host_type",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=["workstation", "workstation", "workstation", "laptop", "laptop", 
                                "server", "server", "iot_device", "printer"],
                        with_replacement=True,
                    ),
                ),
            ),
            Column(
                name="os_type",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=["Windows", "Windows", "Windows", "Linux", "macOS", "Linux"],
                        with_replacement=True,
                    ),
                ),
            ),
            # Foreign key to primary user (can be null for servers)
            Column(
                name="fk_primary_user_id",
                data_type="string",
                column_type=ColumnType.FOREIGN_KEY,
                column_category_type=ColumnCategoryType.METADATA,
            ),
        ],
    )
    
    # ==========================================================================
    # ENTITY 3: external_destination
    # External hosts/IPs that internal hosts connect to
    # ==========================================================================
    external_dest = Entity(
        name="external_destination",
        cardinality=100,  # Pool of external destinations
        columns=[
            Column(
                name="dest_id",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.ID,
                    params=IDParams(template_str="EXT_{id}"),
                ),
            ),
            Column(
                name="domain_name",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=[
                            "google.com", "microsoft.com", "github.com", "aws.amazon.com",
                            "office365.com", "salesforce.com", "slack.com", "zoom.us",
                            "cloudflare.com", "akamai.com", "fastly.com", "dropbox.com",
                            "linkedin.com", "twitter.com", "facebook.com", "youtube.com",
                            "suspicious-domain.xyz", "malware-c2.net", "phishing-site.com",  # Suspicious
                            "unknown-service.io", "api.internal-tool.com", "cdn.example.com",
                        ],
                        with_replacement=True,
                    ),
                ),
            ),
            Column(
                name="ext_ip_octet_1",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=[8, 13, 17, 20, 34, 35, 52, 54, 64, 72, 93, 104, 142, 151, 199, 204, 208, 216],
                        with_replacement=True,
                    ),
                ),
            ),
            Column(
                name="ext_ip_octet_2",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.UNIFORM_DIST,
                    params=UniformDistParams(lower=0, upper=255),
                ),
            ),
            Column(
                name="ext_ip_octet_3",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.UNIFORM_DIST,
                    params=UniformDistParams(lower=0, upper=255),
                ),
            ),
            Column(
                name="ext_ip_octet_4",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.UNIFORM_DIST,
                    params=UniformDistParams(lower=1, upper=254),
                ),
            ),
            Column(
                name="category",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=["business", "business", "cloud_service", "cdn", "social_media", 
                                "streaming", "suspicious", "uncategorized"],
                        with_replacement=True,
                    ),
                ),
            ),
            Column(
                name="risk_score",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=[0, 0, 0, 10, 10, 20, 30, 50, 80, 100],  # Most are low risk
                        with_replacement=True,
                    ),
                ),
            ),
        ],
    )
    
    # ==========================================================================
    # ENTITY 4: service_definition
    # Network services with protocol/port definitions
    # ==========================================================================
    service_def = Entity(
        name="service_definition",
        cardinality=15,
        columns=[
            Column(
                name="service_id",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.ID,
                    params=IDParams(template_str="SVC_{id}"),
                ),
            ),
            Column(
                name="service_name",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=[
                            "HTTP", "HTTPS", "DNS", "SSH", "RDP", "SMB", "SMTP",
                            "LDAP", "MYSQL", "MSSQL", "FTP", "TELNET", "SNMP", "NTP", "ICMP",
                        ],
                        with_replacement=False,
                    ),
                ),
            ),
            Column(
                name="protocol",
                data_type="string",
                column_type=ColumnType.DERIVED,
                column_category_type=ColumnCategoryType.METADATA,
                derivation=Derivation(
                    function_type=DerivationFunctionType.MAP_VALUES,
                    dependent_columns=["service_name"],
                    params=MapValuesParams(
                        mapping=[
                            {"from": "HTTP", "to": "TCP"},
                            {"from": "HTTPS", "to": "TCP"},
                            {"from": "DNS", "to": "UDP"},
                            {"from": "SSH", "to": "TCP"},
                            {"from": "RDP", "to": "TCP"},
                            {"from": "SMB", "to": "TCP"},
                            {"from": "SMTP", "to": "TCP"},
                            {"from": "LDAP", "to": "TCP"},
                            {"from": "MYSQL", "to": "TCP"},
                            {"from": "MSSQL", "to": "TCP"},
                            {"from": "FTP", "to": "TCP"},
                            {"from": "TELNET", "to": "TCP"},
                            {"from": "SNMP", "to": "UDP"},
                            {"from": "NTP", "to": "UDP"},
                            {"from": "ICMP", "to": "ICMP"},
                        ],
                        default="TCP",
                    ),
                ),
            ),
            Column(
                name="port",
                data_type="int64",
                column_type=ColumnType.DERIVED,
                column_category_type=ColumnCategoryType.METADATA,
                derivation=Derivation(
                    function_type=DerivationFunctionType.MAP_VALUES,
                    dependent_columns=["service_name"],
                    params=MapValuesParams(
                        mapping=[
                            {"from": "HTTP", "to": "80"},
                            {"from": "HTTPS", "to": "443"},
                            {"from": "DNS", "to": "53"},
                            {"from": "SSH", "to": "22"},
                            {"from": "RDP", "to": "3389"},
                            {"from": "SMB", "to": "445"},
                            {"from": "SMTP", "to": "25"},
                            {"from": "LDAP", "to": "389"},
                            {"from": "MYSQL", "to": "3306"},
                            {"from": "MSSQL", "to": "1433"},
                            {"from": "FTP", "to": "21"},
                            {"from": "TELNET", "to": "23"},
                            {"from": "SNMP", "to": "161"},
                            {"from": "NTP", "to": "123"},
                            {"from": "ICMP", "to": "0"},
                        ],
                        default="80",
                    ),
                ),
            ),
        ],
    )
    
    # ==========================================================================
    # ENTITY 5: network_session (STATE MACHINE)
    # The central entity that drives all correlated network logs
    # ==========================================================================
    network_session = Entity(
        name="network_session",
        cardinality=n_sessions,
        timestamp=Timestamp(column_name="timestamp"),
        columns=[
            Column(
                name="session_id",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.ID,
                    params=IDParams(template_str="SESS_{id}"),
                ),
            ),
            # Source host
            Column(
                name="fk_src_host_id",
                data_type="string",
                column_type=ColumnType.FOREIGN_KEY,
                column_category_type=ColumnCategoryType.METADATA,
            ),
            # Destination (external)
            Column(
                name="fk_dest_id",
                data_type="string",
                column_type=ColumnType.FOREIGN_KEY,
                column_category_type=ColumnCategoryType.METADATA,
            ),
            # Service type
            Column(
                name="fk_service_id",
                data_type="string",
                column_type=ColumnType.FOREIGN_KEY,
                column_category_type=ColumnCategoryType.METADATA,
            ),
            # Source port (ephemeral)
            Column(
                name="src_port",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.UNIFORM_DIST,
                    params=UniformDistParams(lower=49152, upper=65535),
                ),
            ),
            # Session state machine
            Column(
                name="session_state",
                data_type="string",
                column_type=ColumnType.STATEFUL,
                column_category_type=ColumnCategoryType.MEASUREMENT,
                domain=Domain(
                    type=DomainType.STATE_MACHINE,
                    params=StateMachineParams(
                        column_name="session_state",
                        trigger_column_name="action_type",
                        initial_state="INIT",
                        states=[
                            "INIT",
                            "DNS_LOOKUP",
                            "CONNECTING",
                            "ESTABLISHED",
                            "DATA_TRANSFER",
                            "CLOSING",
                            "CLOSED",
                            "BLOCKED",
                            "ALERTED",
                        ],
                        terminal_states=["CLOSED", "BLOCKED"],
                        context_variables={"is_suspicious": False, "data_transferred": False},
                        transitions=[
                            # Normal flow: INIT -> DNS -> CONNECT -> ESTABLISHED -> DATA -> CLOSE
                            Transition(
                                trigger="dns_query",
                                source="INIT",
                                dest="DNS_LOOKUP",
                                probability=0.7,  # 70% need DNS lookup
                            ),
                            Transition(
                                trigger="direct_connect",
                                source="INIT",
                                dest="CONNECTING",
                                probability=0.3,  # 30% direct IP connection
                            ),
                            Transition(
                                trigger="dns_success",
                                source="DNS_LOOKUP",
                                dest="CONNECTING",
                                probability=0.95,
                            ),
                            Transition(
                                trigger="dns_failure",
                                source="DNS_LOOKUP",
                                dest="CLOSED",
                                probability=0.05,
                            ),
                            Transition(
                                trigger="syn_ack",
                                source="CONNECTING",
                                dest="ESTABLISHED",
                                probability=0.90,
                            ),
                            Transition(
                                trigger="fw_block",
                                source="CONNECTING",
                                dest="BLOCKED",
                                probability=0.05,
                            ),
                            Transition(
                                trigger="timeout",
                                source="CONNECTING",
                                dest="CLOSED",
                                probability=0.05,
                            ),
                            Transition(
                                trigger="data_send",
                                source="ESTABLISHED",
                                dest="DATA_TRANSFER",
                                probability=0.85,
                                context_updates={"data_transferred": True},
                            ),
                            Transition(
                                trigger="ids_alert",
                                source="ESTABLISHED",
                                dest="ALERTED",
                                probability=0.05,
                            ),
                            Transition(
                                trigger="quick_close",
                                source="ESTABLISHED",
                                dest="CLOSING",
                                probability=0.10,
                            ),
                            Transition(
                                trigger="more_data",
                                source="DATA_TRANSFER",
                                dest="DATA_TRANSFER",
                                probability=0.60,
                            ),
                            Transition(
                                trigger="ids_detect",
                                source="DATA_TRANSFER",
                                dest="ALERTED",
                                probability=0.03,
                            ),
                            Transition(
                                trigger="finish",
                                source="DATA_TRANSFER",
                                dest="CLOSING",
                                probability=0.37,
                            ),
                            Transition(
                                trigger="continue_monitored",
                                source="ALERTED",
                                dest="DATA_TRANSFER",
                                probability=0.60,
                            ),
                            Transition(
                                trigger="ids_block",
                                source="ALERTED",
                                dest="BLOCKED",
                                probability=0.40,
                            ),
                            Transition(
                                trigger="fin",
                                source="CLOSING",
                                dest="CLOSED",
                                probability=1.0,
                            ),
                        ],
                    ),
                ),
            ),
            # Traffic metrics
            Column(
                name="bytes_sent",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.EXPONENTIAL_DIST,
                    params=ExponentialDistParams(scale=50000),
                ),
            ),
            Column(
                name="bytes_received",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.EXPONENTIAL_DIST,
                    params=ExponentialDistParams(scale=100000),
                ),
            ),
            Column(
                name="packets",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.EXPONENTIAL_DIST,
                    params=ExponentialDistParams(scale=100),
                ),
            ),
            Column(
                name="duration_ms",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.EXPONENTIAL_DIST,
                    params=ExponentialDistParams(scale=30000),
                ),
            ),
        ],
    )
    
    # ==========================================================================
    # ENTITY 6: endpoint_event (EDR)
    # Host-based events: process, file, registry, network
    # ==========================================================================
    endpoint_event = Entity(
        name="endpoint_event",
        cardinality=n_endpoint_events,
        timestamp=Timestamp(column_name="timestamp"),
        columns=[
            Column(
                name="event_id",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.ID,
                    params=IDParams(template_str="EDR_{id}"),
                ),
            ),
            Column(
                name="fk_host_id",
                data_type="string",
                column_type=ColumnType.FOREIGN_KEY,
                column_category_type=ColumnCategoryType.METADATA,
            ),
            Column(
                name="event_type",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=[
                            "process_start", "process_start", "process_start",
                            "process_stop", "process_stop",
                            "file_create", "file_modify", "file_delete",
                            "registry_modify",
                            "network_connect", "network_connect",
                            "dll_load", "dll_load",
                        ],
                        with_replacement=True,
                    ),
                ),
            ),
            Column(
                name="process_name",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=[
                            "chrome.exe", "chrome.exe", "firefox.exe", "msedge.exe",
                            "outlook.exe", "teams.exe", "slack.exe", "zoom.exe",
                            "explorer.exe", "svchost.exe", "svchost.exe", "csrss.exe",
                            "powershell.exe", "cmd.exe", "python.exe", "node.exe",
                            "winword.exe", "excel.exe", "notepad.exe",
                            "suspicious.exe", "unknown_process.exe",  # Potentially suspicious
                        ],
                        with_replacement=True,
                    ),
                ),
            ),
            Column(
                name="process_path",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=[
                            "C:\\Program Files\\Google\\Chrome\\",
                            "C:\\Program Files\\Mozilla Firefox\\",
                            "C:\\Program Files (x86)\\Microsoft\\Edge\\",
                            "C:\\Program Files\\Microsoft Office\\",
                            "C:\\Windows\\System32\\",
                            "C:\\Windows\\System32\\",
                            "C:\\Users\\user\\AppData\\Local\\",
                            "C:\\Users\\user\\Downloads\\",  # Suspicious location
                            "C:\\Temp\\",  # Suspicious location
                        ],
                        with_replacement=True,
                    ),
                ),
            ),
            Column(
                name="parent_process",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=[
                            "explorer.exe", "explorer.exe", "explorer.exe",
                            "svchost.exe", "svchost.exe",
                            "services.exe", "winlogon.exe",
                            "cmd.exe", "powershell.exe",  # Potentially suspicious parent
                        ],
                        with_replacement=True,
                    ),
                ),
            ),
            Column(
                name="severity",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=["info", "info", "info", "info", "low", "low", "medium", "high", "critical"],
                        with_replacement=True,
                    ),
                ),
            ),
            Column(
                name="process_hash",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.ID,
                    params=IDParams(template_str="SHA256_{id}"),
                ),
            ),
            # State machine for endpoint event lifecycle
            Column(
                name="event_state",
                data_type="string",
                column_type=ColumnType.STATEFUL,
                column_category_type=ColumnCategoryType.MEASUREMENT,
                domain=Domain(
                    type=DomainType.STATE_MACHINE,
                    params=StateMachineParams(
                        column_name="event_state",
                        trigger_column_name="event_action",
                        initial_state="DETECTED",
                        states=[
                            "DETECTED",
                            "ANALYZING",
                            "QUARANTINED",
                            "REMEDIATED",
                            "ESCALATED",
                            "RESOLVED",
                            "IGNORED",
                        ],
                        terminal_states=["RESOLVED", "IGNORED"],
                        transitions=[
                            Transition(
                                trigger="analyze",
                                source="DETECTED",
                                dest="ANALYZING",
                                probability=0.7,
                            ),
                            Transition(
                                trigger="ignore_low",
                                source="DETECTED",
                                dest="IGNORED",
                                probability=0.3,
                            ),
                            Transition(
                                trigger="quarantine",
                                source="ANALYZING",
                                dest="QUARANTINED",
                                probability=0.3,
                            ),
                            Transition(
                                trigger="escalate",
                                source="ANALYZING",
                                dest="ESCALATED",
                                probability=0.2,
                            ),
                            Transition(
                                trigger="resolve_clean",
                                source="ANALYZING",
                                dest="RESOLVED",
                                probability=0.5,
                            ),
                            Transition(
                                trigger="remediate",
                                source="QUARANTINED",
                                dest="REMEDIATED",
                                probability=0.8,
                            ),
                            Transition(
                                trigger="escalate_quarantine",
                                source="QUARANTINED",
                                dest="ESCALATED",
                                probability=0.2,
                            ),
                            Transition(
                                trigger="close_remediated",
                                source="REMEDIATED",
                                dest="RESOLVED",
                                probability=1.0,
                            ),
                            Transition(
                                trigger="close_escalated",
                                source="ESCALATED",
                                dest="RESOLVED",
                                probability=1.0,
                            ),
                        ],
                    ),
                ),
            ),
        ],
    )
    
    # ==========================================================================
    # ENTITY 7: vpn_event
    # VPN connection events for remote access
    # ==========================================================================
    vpn_event = Entity(
        name="vpn_event",
        cardinality=n_vpn_events,
        timestamp=Timestamp(column_name="timestamp"),
        columns=[
            Column(
                name="vpn_event_id",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.ID,
                    params=IDParams(template_str="VPN_{id}"),
                ),
            ),
            Column(
                name="fk_user_id",
                data_type="string",
                column_type=ColumnType.FOREIGN_KEY,
                column_category_type=ColumnCategoryType.METADATA,
            ),
            Column(
                name="event_type",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=["connect", "connect", "disconnect", "disconnect", 
                                "auth_success", "auth_failure", "session_timeout"],
                        with_replacement=True,
                    ),
                ),
            ),
            # External source IP octets
            Column(
                name="source_ip_octet_1",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=[24, 47, 68, 72, 98, 108, 136, 142, 165, 173, 184, 192, 199, 203, 207, 216],
                        with_replacement=True,
                    ),
                ),
            ),
            Column(
                name="source_ip_octet_2",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.UNIFORM_DIST,
                    params=UniformDistParams(lower=0, upper=255),
                ),
            ),
            Column(
                name="source_ip_octet_3",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.UNIFORM_DIST,
                    params=UniformDistParams(lower=0, upper=255),
                ),
            ),
            Column(
                name="source_ip_octet_4",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.UNIFORM_DIST,
                    params=UniformDistParams(lower=1, upper=254),
                ),
            ),
            Column(
                name="tunnel_type",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=["SSL", "SSL", "IPSec", "L2TP", "WireGuard"],
                        with_replacement=True,
                    ),
                ),
            ),
            Column(
                name="geo_country",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=["US", "US", "US", "US", "US", "CA", "UK", "DE", "IN", "AU", "CN", "RU"],
                        with_replacement=True,
                    ),
                ),
            ),
            Column(
                name="duration_seconds",
                data_type="int64",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.EXPONENTIAL_DIST,
                    params=ExponentialDistParams(scale=7200),  # Mean ~2 hours
                ),
            ),
            # State machine for VPN connection lifecycle
            Column(
                name="connection_state",
                data_type="string",
                column_type=ColumnType.STATEFUL,
                column_category_type=ColumnCategoryType.MEASUREMENT,
                domain=Domain(
                    type=DomainType.STATE_MACHINE,
                    params=StateMachineParams(
                        column_name="connection_state",
                        trigger_column_name="vpn_action",
                        initial_state="INITIATING",
                        states=[
                            "INITIATING",
                            "AUTHENTICATING",
                            "CONNECTED",
                            "ACTIVE",
                            "IDLE",
                            "DISCONNECTING",
                            "DISCONNECTED",
                            "FAILED",
                        ],
                        terminal_states=["DISCONNECTED", "FAILED"],
                        transitions=[
                            Transition(
                                trigger="start_auth",
                                source="INITIATING",
                                dest="AUTHENTICATING",
                                probability=1.0,
                            ),
                            Transition(
                                trigger="auth_success",
                                source="AUTHENTICATING",
                                dest="CONNECTED",
                                probability=0.85,
                            ),
                            Transition(
                                trigger="auth_fail",
                                source="AUTHENTICATING",
                                dest="FAILED",
                                probability=0.15,
                            ),
                            Transition(
                                trigger="start_traffic",
                                source="CONNECTED",
                                dest="ACTIVE",
                                probability=0.9,
                            ),
                            Transition(
                                trigger="quick_disconnect",
                                source="CONNECTED",
                                dest="DISCONNECTING",
                                probability=0.1,
                            ),
                            Transition(
                                trigger="go_idle",
                                source="ACTIVE",
                                dest="IDLE",
                                probability=0.4,
                            ),
                            Transition(
                                trigger="user_disconnect",
                                source="ACTIVE",
                                dest="DISCONNECTING",
                                probability=0.6,
                            ),
                            Transition(
                                trigger="resume_traffic",
                                source="IDLE",
                                dest="ACTIVE",
                                probability=0.5,
                            ),
                            Transition(
                                trigger="timeout_disconnect",
                                source="IDLE",
                                dest="DISCONNECTING",
                                probability=0.5,
                            ),
                            Transition(
                                trigger="complete_disconnect",
                                source="DISCONNECTING",
                                dest="DISCONNECTED",
                                probability=1.0,
                            ),
                        ],
                    ),
                ),
            ),
        ],
    )
    
    # ==========================================================================
    # ENTITY 8: auth_event
    # IAM/Authentication events
    # ==========================================================================
    auth_event = Entity(
        name="auth_event",
        cardinality=n_auth_events,
        timestamp=Timestamp(column_name="timestamp"),
        columns=[
            Column(
                name="auth_event_id",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.ID,
                    params=IDParams(template_str="AUTH_{id}"),
                ),
            ),
            Column(
                name="fk_user_id",
                data_type="string",
                column_type=ColumnType.FOREIGN_KEY,
                column_category_type=ColumnCategoryType.METADATA,
            ),
            Column(
                name="fk_host_id",
                data_type="string",
                column_type=ColumnType.FOREIGN_KEY,
                column_category_type=ColumnCategoryType.METADATA,
            ),
            Column(
                name="event_type",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=[
                            "login_success", "login_success", "login_success", "login_success",
                            "login_failure", "login_failure",
                            "logout", "logout",
                            "password_change", "privilege_escalation", "account_lockout",
                        ],
                        with_replacement=True,
                    ),
                ),
            ),
            Column(
                name="auth_method",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=["password", "password", "mfa", "mfa", "certificate", "sso", "kerberos"],
                        with_replacement=True,
                    ),
                ),
            ),
            Column(
                name="target_system",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=["Workstation", "Workstation", "VPN", "Email", "Application", "Server", "Database"],
                        with_replacement=True,
                    ),
                ),
            ),
            Column(
                name="failure_reason",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=["none", "none", "none", "none", "invalid_password", "account_disabled", 
                                "mfa_timeout", "expired_password", "account_locked"],
                        with_replacement=True,
                    ),
                ),
            ),
            Column(
                name="is_anomalous",
                data_type="string",
                column_type=ColumnType.INDEPENDENT,
                column_category_type=ColumnCategoryType.METADATA,
                domain=Domain(
                    type=DomainType.CATEGORICAL,
                    params=CategoricalParams(
                        values=["false", "false", "false", "false", "false", "false", "false", "false", "true"],
                        with_replacement=True,
                    ),
                ),
            ),
            # State machine for authentication session lifecycle
            Column(
                name="auth_state",
                data_type="string",
                column_type=ColumnType.STATEFUL,
                column_category_type=ColumnCategoryType.MEASUREMENT,
                domain=Domain(
                    type=DomainType.STATE_MACHINE,
                    params=StateMachineParams(
                        column_name="auth_state",
                        trigger_column_name="auth_action",
                        initial_state="ATTEMPT",
                        states=[
                            "ATTEMPT",
                            "MFA_PENDING",
                            "VERIFIED",
                            "SESSION_ACTIVE",
                            "SESSION_IDLE",
                            "LOGGED_OUT",
                            "DENIED",
                            "LOCKED",
                        ],
                        terminal_states=["LOGGED_OUT", "DENIED", "LOCKED"],
                        transitions=[
                            Transition(
                                trigger="password_ok",
                                source="ATTEMPT",
                                dest="MFA_PENDING",
                                probability=0.6,
                            ),
                            Transition(
                                trigger="single_factor_ok",
                                source="ATTEMPT",
                                dest="VERIFIED",
                                probability=0.25,
                            ),
                            Transition(
                                trigger="auth_denied",
                                source="ATTEMPT",
                                dest="DENIED",
                                probability=0.1,
                            ),
                            Transition(
                                trigger="account_lock",
                                source="ATTEMPT",
                                dest="LOCKED",
                                probability=0.05,
                            ),
                            Transition(
                                trigger="mfa_success",
                                source="MFA_PENDING",
                                dest="VERIFIED",
                                probability=0.85,
                            ),
                            Transition(
                                trigger="mfa_fail",
                                source="MFA_PENDING",
                                dest="DENIED",
                                probability=0.15,
                            ),
                            Transition(
                                trigger="create_session",
                                source="VERIFIED",
                                dest="SESSION_ACTIVE",
                                probability=1.0,
                            ),
                            Transition(
                                trigger="go_idle",
                                source="SESSION_ACTIVE",
                                dest="SESSION_IDLE",
                                probability=0.4,
                            ),
                            Transition(
                                trigger="user_logout",
                                source="SESSION_ACTIVE",
                                dest="LOGGED_OUT",
                                probability=0.6,
                            ),
                            Transition(
                                trigger="resume_activity",
                                source="SESSION_IDLE",
                                dest="SESSION_ACTIVE",
                                probability=0.5,
                            ),
                            Transition(
                                trigger="session_timeout",
                                source="SESSION_IDLE",
                                dest="LOGGED_OUT",
                                probability=0.5,
                            ),
                        ],
                    ),
                ),
            ),
        ],
    )
    
    # ==========================================================================
    # ENTITY RELATIONSHIPS
    # ==========================================================================
    relationships = [
        # User -> Host (one-to-many: user can have multiple hosts)
        EntityRelationship(
            parent_entity="user",
            child_entity="host",
            relationship_type=EntityRelationshipType.ONE_TO_MANY,
            join_columns={"user_id": "fk_primary_user_id"},
        ),
        # Host -> Network Session (one-to-many)
        EntityRelationship(
            parent_entity="host",
            child_entity="network_session",
            relationship_type=EntityRelationshipType.ONE_TO_MANY,
            join_columns={"host_id": "fk_src_host_id"},
        ),
        # External Destination -> Network Session (one-to-many)
        EntityRelationship(
            parent_entity="external_destination",
            child_entity="network_session",
            relationship_type=EntityRelationshipType.ONE_TO_MANY,
            join_columns={"dest_id": "fk_dest_id"},
        ),
        # Service Definition -> Network Session (one-to-many)
        EntityRelationship(
            parent_entity="service_definition",
            child_entity="network_session",
            relationship_type=EntityRelationshipType.ONE_TO_MANY,
            join_columns={"service_id": "fk_service_id"},
        ),
        # Host -> Endpoint Event (one-to-many)
        EntityRelationship(
            parent_entity="host",
            child_entity="endpoint_event",
            relationship_type=EntityRelationshipType.ONE_TO_MANY,
            join_columns={"host_id": "fk_host_id"},
        ),
        # User -> VPN Event (one-to-many)
        EntityRelationship(
            parent_entity="user",
            child_entity="vpn_event",
            relationship_type=EntityRelationshipType.ONE_TO_MANY,
            join_columns={"user_id": "fk_user_id"},
        ),
        # User -> Auth Event (one-to-many)
        EntityRelationship(
            parent_entity="user",
            child_entity="auth_event",
            relationship_type=EntityRelationshipType.ONE_TO_MANY,
            join_columns={"user_id": "fk_user_id"},
        ),
        # Host -> Auth Event (one-to-many)
        EntityRelationship(
            parent_entity="host",
            child_entity="auth_event",
            relationship_type=EntityRelationshipType.ONE_TO_MANY,
            join_columns={"host_id": "fk_host_id"},
        ),
    ]
    
    # ==========================================================================
    # GLOBAL TIMESTAMP
    # ==========================================================================
    global_ts = GlobalTimestamp(
        t_start="2025-02-01T00:00:00+00:00",
        t_end="2025-02-01T23:59:00+00:00",
        time_interval="1min",
    )
    
    return DataSchema(
        entities=[user, host, external_dest, service_def, network_session, 
                  endpoint_event, vpn_event, auth_event],
        entity_relationships=relationships,
        global_timestamp=global_ts,
    )

In [5]:
# Create the schema instance
siem_schema = create_siem_edr_schema(
    n_hosts=N_HOSTS,
    n_users=N_USERS,
    n_sessions=N_SESSIONS,
    n_vpn_events=N_VPN_EVENTS,
    n_endpoint_events=N_ENDPOINT_EVENTS,
    n_auth_events=N_AUTH_EVENTS,
)

print(f"Schema created with {len(siem_schema.entities)} entities:")
for entity in siem_schema.entities:
    print(f"  - {entity.name}: {entity.cardinality} rows")

Schema created with 8 entities:
  - user: 150 rows
  - host: 200 rows
  - external_destination: 100 rows
  - service_definition: 15 rows
  - network_session: 5000 rows
  - endpoint_event: 10000 rows
  - vpn_event: 500 rows
  - auth_event: 3000 rows


## Run Data Generation

In [6]:
config = ra.GenerateFromDataSchema.Config(
    schema=siem_schema,
    upload_datasets=True,
)
generate = ra.GenerateFromDataSchema(config)

In [7]:
builder = rf.WorkflowBuilder()
builder.add(generate)
workflow = await builder.start(conn)
print(f"Workflow ID: {workflow.id()}")

Workflow ID: 5RDz6J5NZrbvpeDjeJBYD6


In [8]:
async for log in workflow.logs(level=rf.events.LogLevel.DEBUG):
    print(log)

2026-02-25T23:20:03.695558Z generate-from-data-schema: INFO Generating 8 entities: user, host, external_destination, service_definition, network_session, endpoint_event, vpn_event, auth_event
2026-02-25T23:20:03.706632Z generate-from-data-schema: INFO Starting data generation...
2026-02-25T23:21:51.857143Z generate-from-data-schema: INFO Generated 8 entity tables
2026-02-25T23:21:51.882264Z generate-from-data-schema: INFO Creating dataset for entity 'user': 150 rows
2026-02-25T23:21:52.123938Z generate-from-data-schema: INFO Uploaded dataset 'user' (5Bax2GzxafZ5CInwG8ZtUv): 150 rows
2026-02-25T23:21:52.144196Z generate-from-data-schema: INFO Creating dataset for entity 'external_destination': 100 rows
2026-02-25T23:21:52.305479Z generate-from-data-schema: INFO Uploaded dataset 'external_destination' (6q98f9IVTIaxaOeksigr8S): 100 rows
2026-02-25T23:21:52.327393Z generate-from-data-schema: INFO Creating dataset for entity 'service_definition': 15 rows
2026-02-25T23:21:52.516779Z generate

## Retrieve Generated Datasets

In [9]:
datasets = await workflow.datasets().collect()
print(f"Generated {len(datasets)} datasets")

# Store datasets by name
dataset_dict = {}
for remote_ds in datasets:
    ds = await remote_ds.to_local(conn)
    dataset_dict[ds.name()] = ds
    print(f"  - {ds.name()}: {ds.table.num_rows} rows")

Generated 8 datasets
  - user: 150 rows
  - external_destination: 100 rows
  - service_definition: 15 rows
  - host: 200 rows
  - vpn_event: 3008 rows
  - network_session: 37338 rows
  - endpoint_event: 32619 rows
  - auth_event: 14127 rows


## Explore Generated Data

In [10]:
# Users
user_df = dataset_dict["user"].to_pandas()
print(f"Users: {len(user_df)} records")
print("\nDepartment Distribution:")
print(user_df["department"].value_counts())
print("\nRole Distribution:")
print(user_df["role"].value_counts())
user_df.head(10)

Users: 150 records

Department Distribution:
department
IT             39
Engineering    34
Executive      23
Sales          21
HR             20
Finance        13
Name: count, dtype: int64

Role Distribution:
role
employee           77
service_account    28
contractor         23
admin              22
Name: count, dtype: int64


,user_id,username,department,role,is_privileged
0,USER_0,user0,IT,admin,false
1,USER_1,user1,Executive,employee,true
2,USER_2,user2,Engineering,employee,false
3,USER_3,user3,Sales,admin,true
4,USER_4,user4,HR,employee,false
5,USER_5,user5,HR,employee,false
6,USER_6,user6,Finance,service_account,false
7,USER_7,user7,Engineering,contractor,false
8,USER_8,user8,Executive,employee,false
9,USER_9,user9,HR,employee,false


In [11]:
# Hosts
host_df = dataset_dict["host"].to_pandas()
print(f"Hosts: {len(host_df)} records")

# Construct IP addresses
host_df['ip_address'] = (
    host_df['ip_octet_1'].astype(str) + '.' +
    host_df['ip_octet_2'].astype(str) + '.' +
    host_df['ip_octet_3'].astype(str) + '.' +
    host_df['ip_octet_4'].astype(str)
)

print("\nHost Type Distribution:")
print(host_df["host_type"].value_counts())
print("\nOS Distribution:")
print(host_df["os_type"].value_counts())
host_df[['host_id', 'hostname', 'ip_address', 'host_type', 'os_type', 'segment']].head(10)

Hosts: 200 records

Host Type Distribution:
host_type
workstation    71
server         52
laptop         38
iot_device     22
printer        17
Name: count, dtype: int64

OS Distribution:
os_type
Windows    102
Linux       59
macOS       39
Name: count, dtype: int64


,host_id,hostname,ip_address,host_type,os_type,segment
0,HOST_0,host0,10.1.9.46,laptop,Windows,dmz
1,HOST_1,host1,10.3.156.237,laptop,Windows,segment_b
2,HOST_2,host2,10.3.253.114,server,Windows,segment_a
3,HOST_3,host3,10.1.186.84,laptop,Windows,segment_a
4,HOST_4,host4,10.2.112.148,server,macOS,segment_b
5,HOST_5,host5,172.16.2.119,workstation,Linux,segment_c
6,HOST_6,host6,172.2.249.15,workstation,macOS,segment_b
7,HOST_7,host7,172.2.143.213,server,Linux,segment_b
8,HOST_8,host8,172.1.34.215,server,Windows,segment_b
9,HOST_9,host9,10.2.8.219,iot_device,Linux,dmz


In [12]:
# Network Sessions
session_df = dataset_dict["network_session"].to_pandas()
print(f"Network Sessions: {len(session_df)} records")
print(f"Unique sessions: {session_df['session_id'].nunique()}")

print("\nSession State Distribution:")
print(session_df["session_state"].value_counts())
print("\nAction Type Distribution:")
print(session_df["action_type"].value_counts())
session_df.head(15)

Network Sessions: 37338 records
Unique sessions: 5000

Session State Distribution:
session_state
DATA_TRANSFER    10072
INIT              5000
CONNECTING        4822
CLOSED            4512
ESTABLISHED       4315
CLOSING           4089
DNS_LOOKUP        3532
ALERTED            508
BLOCKED            488
Name: count, dtype: int64

Action Type Distribution:
action_type
more_data             6071
syn_ack               4315
fin                   4089
data_send             3719
finish                3702
dns_query             3532
dns_success           3354
direct_connect        1468
quick_close            387
ids_detect             299
continue_monitored     282
fw_block               262
timeout                245
ids_block              226
ids_alert              209
dns_failure            178
Name: count, dtype: int64


,session_id,fk_src_host_id,fk_dest_id,fk_service_id,src_port,session_state,action_type,data_transferred,is_suspicious,bytes_sent,bytes_received,packets,duration_ms,timestamp
0,SESS_0,HOST_170,EXT_85,SVC_12,55405,INIT,None,False,False,16611,257162,33,31720,2025-02-01T00:00:00+00:00
1,SESS_0,HOST_170,EXT_85,SVC_12,55405,DNS_LOOKUP,dns_query,False,False,16611,257162,33,31720,2025-02-01T00:01:00+00:00
2,SESS_0,HOST_170,EXT_85,SVC_12,55405,CONNECTING,dns_success,False,False,16611,257162,33,31720,2025-02-01T00:02:00+00:00
3,SESS_0,HOST_170,EXT_85,SVC_12,55405,ESTABLISHED,syn_ack,False,False,16611,257162,33,31720,2025-02-01T00:03:00+00:00
4,SESS_0,HOST_170,EXT_85,SVC_12,55405,DATA_TRANSFER,data_send,True,False,16611,257162,33,31720,2025-02-01T00:04:00+00:00
5,SESS_0,HOST_170,EXT_85,SVC_12,55405,CLOSING,finish,True,False,16611,257162,33,31720,2025-02-01T00:05:00+00:00
6,SESS_0,HOST_170,EXT_85,SVC_12,55405,CLOSED,fin,True,False,16611,257162,33,31720,2025-02-01T00:06:00+00:00
7,SESS_1,HOST_127,EXT_63,SVC_9,57685,INIT,None,False,False,64237,268609,70,71566,2025-02-01T00:00:00+00:00
8,SESS_1,HOST_127,EXT_63,SVC_9,57685,DNS_LOOKUP,dns_query,False,False,64237,268609,70,71566,2025-02-01T00:01:00+00:00
9,SESS_1,HOST_127,EXT_63,SVC_9,57685,CLOSED,dns_failure,False,False,64237,268609,70,71566,2025-02-01T00:02:00+00:00


In [13]:
# Endpoint Events (EDR)
endpoint_df = dataset_dict["endpoint_event"].to_pandas()
print(f"Endpoint Events: {len(endpoint_df)} records")

print("\nEvent Type Distribution:")
print(endpoint_df["event_type"].value_counts())
print("\nSeverity Distribution:")
print(endpoint_df["severity"].value_counts())
print("\nTop Processes:")
print(endpoint_df["process_name"].value_counts().head(10))
endpoint_df.head(10)

Endpoint Events: 32619 records

Event Type Distribution:
event_type
process_start      7255
process_stop       5342
network_connect    5029
dll_load           4991
file_delete        2582
registry_modify    2520
file_modify        2492
file_create        2408
Name: count, dtype: int64

Severity Distribution:
severity
info        14442
low          7313
medium       3788
critical     3625
high         3451
Name: count, dtype: int64

Top Processes:
process_name
svchost.exe       3170
chrome.exe        3109
suspicious.exe    1677
python.exe        1643
msedge.exe        1630
node.exe          1605
winword.exe       1600
csrss.exe         1588
firefox.exe       1585
slack.exe         1576
Name: count, dtype: int64


,event_id,fk_host_id,event_type,process_name,process_path,parent_process,severity,process_hash,event_state,event_action,timestamp
0,EDR_0,HOST_170,process_stop,firefox.exe,C:\Program Files\Mozilla Firefox\,explorer.exe,info,SHA256_0,DETECTED,None,2025-02-01T00:00:00+00:00
1,EDR_0,HOST_170,process_stop,firefox.exe,C:\Program Files\Mozilla Firefox\,explorer.exe,info,SHA256_0,ANALYZING,analyze,2025-02-01T00:01:00+00:00
2,EDR_0,HOST_170,process_stop,firefox.exe,C:\Program Files\Mozilla Firefox\,explorer.exe,info,SHA256_0,QUARANTINED,quarantine,2025-02-01T00:02:00+00:00
3,EDR_0,HOST_170,process_stop,firefox.exe,C:\Program Files\Mozilla Firefox\,explorer.exe,info,SHA256_0,REMEDIATED,remediate,2025-02-01T00:03:00+00:00
4,EDR_0,HOST_170,process_stop,firefox.exe,C:\Program Files\Mozilla Firefox\,explorer.exe,info,SHA256_0,RESOLVED,close_remediated,2025-02-01T00:04:00+00:00
5,EDR_1,HOST_127,file_modify,explorer.exe,C:\Windows\System32\,services.exe,info,SHA256_1,DETECTED,None,2025-02-01T00:00:00+00:00
6,EDR_1,HOST_127,file_modify,explorer.exe,C:\Windows\System32\,services.exe,info,SHA256_1,ANALYZING,analyze,2025-02-01T00:01:00+00:00
7,EDR_1,HOST_127,file_modify,explorer.exe,C:\Windows\System32\,services.exe,info,SHA256_1,RESOLVED,resolve_clean,2025-02-01T00:02:00+00:00
8,EDR_2,HOST_102,network_connect,svchost.exe,C:\Windows\System32\,powershell.exe,low,SHA256_2,DETECTED,None,2025-02-01T00:00:00+00:00
9,EDR_2,HOST_102,network_connect,svchost.exe,C:\Windows\System32\,powershell.exe,low,SHA256_2,ANALYZING,analyze,2025-02-01T00:01:00+00:00


In [14]:
# VPN Events
vpn_df = dataset_dict["vpn_event"].to_pandas()
print(f"VPN Events: {len(vpn_df)} records")

# Construct source IPs
vpn_df['source_ip'] = (
    vpn_df['source_ip_octet_1'].astype(str) + '.' +
    vpn_df['source_ip_octet_2'].astype(str) + '.' +
    vpn_df['source_ip_octet_3'].astype(str) + '.' +
    vpn_df['source_ip_octet_4'].astype(str)
)

print("\nVPN Event Type Distribution:")
print(vpn_df["event_type"].value_counts())
print("\nGeo Location Distribution:")
print(vpn_df["geo_country"].value_counts())
vpn_df[['vpn_event_id', 'fk_user_id', 'event_type', 'source_ip', 'tunnel_type', 'geo_country']].head(10)

VPN Events: 3008 records

VPN Event Type Distribution:
event_type
connect            935
disconnect         807
session_timeout    434
auth_failure       416
auth_success       416
Name: count, dtype: int64

Geo Location Distribution:
geo_country
US    1295
IN     313
UK     276
RU     241
AU     237
CA     235
CN     229
DE     182
Name: count, dtype: int64


,vpn_event_id,fk_user_id,event_type,source_ip,tunnel_type,geo_country
0,VPN_0,USER_127,disconnect,68.96.106.24,WireGuard,AU
1,VPN_0,USER_127,disconnect,68.96.106.24,WireGuard,AU
2,VPN_0,USER_127,disconnect,68.96.106.24,WireGuard,AU
3,VPN_0,USER_127,disconnect,68.96.106.24,WireGuard,AU
4,VPN_0,USER_127,disconnect,68.96.106.24,WireGuard,AU
5,VPN_0,USER_127,disconnect,68.96.106.24,WireGuard,AU
6,VPN_0,USER_127,disconnect,68.96.106.24,WireGuard,AU
7,VPN_1,USER_95,disconnect,142.51.70.132,IPSec,US
8,VPN_1,USER_95,disconnect,142.51.70.132,IPSec,US
9,VPN_1,USER_95,disconnect,142.51.70.132,IPSec,US


In [15]:
# Authentication Events
auth_df = dataset_dict["auth_event"].to_pandas()
print(f"Auth Events: {len(auth_df)} records")

print("\nAuth Event Type Distribution:")
print(auth_df["event_type"].value_counts())
print("\nAuth Method Distribution:")
print(auth_df["auth_method"].value_counts())
print("\nFailure Reasons (non-null):")
print(auth_df[auth_df["failure_reason"] != "none"]["failure_reason"].value_counts())
auth_df.head(10)

Auth Events: 14127 records

Auth Event Type Distribution:
event_type
login_success           5112
login_failure           2444
logout                  2444
password_change         1422
account_lockout         1399
privilege_escalation    1306
Name: count, dtype: int64

Auth Method Distribution:
auth_method
mfa            4185
password       4121
certificate    1983
sso            1930
kerberos       1908
Name: count, dtype: int64

Failure Reasons (non-null):
failure_reason
mfa_timeout         1578
account_disabled    1573
account_locked      1485
invalid_password    1483
expired_password    1448
Name: count, dtype: int64


,auth_event_id,fk_user_id,fk_host_id,event_type,auth_method,target_system,failure_reason,is_anomalous,auth_state,auth_action,timestamp
0,AUTH_0,USER_127,HOST_170,privilege_escalation,mfa,VPN,mfa_timeout,false,ATTEMPT,None,2025-02-01T00:00:00+00:00
1,AUTH_0,USER_127,HOST_170,privilege_escalation,mfa,VPN,mfa_timeout,false,VERIFIED,single_factor_ok,2025-02-01T00:01:00+00:00
2,AUTH_0,USER_127,HOST_170,privilege_escalation,mfa,VPN,mfa_timeout,false,SESSION_ACTIVE,create_session,2025-02-01T00:02:00+00:00
3,AUTH_0,USER_127,HOST_170,privilege_escalation,mfa,VPN,mfa_timeout,false,SESSION_IDLE,go_idle,2025-02-01T00:03:00+00:00
4,AUTH_0,USER_127,HOST_170,privilege_escalation,mfa,VPN,mfa_timeout,false,SESSION_ACTIVE,resume_activity,2025-02-01T00:04:00+00:00
5,AUTH_0,USER_127,HOST_170,privilege_escalation,mfa,VPN,mfa_timeout,false,LOGGED_OUT,user_logout,2025-02-01T00:05:00+00:00
6,AUTH_1,USER_95,HOST_127,login_failure,mfa,Workstation,none,false,ATTEMPT,None,2025-02-01T00:00:00+00:00
7,AUTH_1,USER_95,HOST_127,login_failure,mfa,Workstation,none,false,MFA_PENDING,password_ok,2025-02-01T00:01:00+00:00
8,AUTH_1,USER_95,HOST_127,login_failure,mfa,Workstation,none,false,DENIED,mfa_fail,2025-02-01T00:02:00+00:00
9,AUTH_2,USER_76,HOST_102,login_success,mfa,Server,none,false,ATTEMPT,None,2025-02-01T00:00:00+00:00


## Derive Correlated Log Types

Now we derive additional log types (Firewall, IDS, DNS, Proxy, Netflow) from the network session data to ensure proper correlation.

In [16]:
# Get external destinations and services for joining
ext_dest_df = dataset_dict["external_destination"].to_pandas()
ext_dest_df['dest_ip'] = (
    ext_dest_df['ext_ip_octet_1'].astype(str) + '.' +
    ext_dest_df['ext_ip_octet_2'].astype(str) + '.' +
    ext_dest_df['ext_ip_octet_3'].astype(str) + '.' +
    ext_dest_df['ext_ip_octet_4'].astype(str)
)

service_df = dataset_dict["service_definition"].to_pandas()
print("External Destinations:")
ext_dest_df.head()

External Destinations:


,dest_id,domain_name,ext_ip_octet_1,ext_ip_octet_2,ext_ip_octet_3,ext_ip_octet_4,category,risk_score,dest_ip
0,EXT_0,youtube.com,64,125,171,51,cdn,10,64.125.171.51
1,EXT_1,suspicious-domain.xyz,64,20,232,251,uncategorized,30,64.20.232.251
2,EXT_2,malware-c2.net,8,89,201,77,cdn,80,8.89.201.77
3,EXT_3,google.com,17,96,131,22,cdn,20,17.96.131.22
4,EXT_4,salesforce.com,34,68,83,127,cdn,100,34.68.83.127


In [17]:
# Join session data with host, external destination, and service
session_full = session_df.merge(
    host_df[['host_id', 'ip_address', 'hostname', 'host_type']],
    left_on='fk_src_host_id',
    right_on='host_id',
    how='left'
).rename(columns={'ip_address': 'src_ip', 'hostname': 'src_hostname'})

session_full = session_full.merge(
    ext_dest_df[['dest_id', 'dest_ip', 'domain_name', 'category', 'risk_score']],
    left_on='fk_dest_id',
    right_on='dest_id',
    how='left'
)

session_full = session_full.merge(
    service_df[['service_id', 'service_name', 'protocol', 'port']],
    left_on='fk_service_id',
    right_on='service_id',
    how='left'
).rename(columns={'port': 'dst_port'})

print(f"Full session data: {len(session_full)} records")
session_full.head()

Full session data: 37338 records


,session_id,fk_src_host_id,fk_dest_id,fk_service_id,src_port,session_state,action_type,data_transferred,is_suspicious,bytes_sent,...,host_type,dest_id,dest_ip,domain_name,category,risk_score,service_id,service_name,protocol,dst_port
0,SESS_0,HOST_170,EXT_85,SVC_12,55405,INIT,None,False,False,16611,...,workstation,EXT_85,13.198.56.18,malware-c2.net,uncategorized,20,SVC_12,NTP,UDP,123
1,SESS_0,HOST_170,EXT_85,SVC_12,55405,DNS_LOOKUP,dns_query,False,False,16611,...,workstation,EXT_85,13.198.56.18,malware-c2.net,uncategorized,20,SVC_12,NTP,UDP,123
2,SESS_0,HOST_170,EXT_85,SVC_12,55405,CONNECTING,dns_success,False,False,16611,...,workstation,EXT_85,13.198.56.18,malware-c2.net,uncategorized,20,SVC_12,NTP,UDP,123
3,SESS_0,HOST_170,EXT_85,SVC_12,55405,ESTABLISHED,syn_ack,False,False,16611,...,workstation,EXT_85,13.198.56.18,malware-c2.net,uncategorized,20,SVC_12,NTP,UDP,123
4,SESS_0,HOST_170,EXT_85,SVC_12,55405,DATA_TRANSFER,data_send,True,False,16611,...,workstation,EXT_85,13.198.56.18,malware-c2.net,uncategorized,20,SVC_12,NTP,UDP,123


In [18]:
# ==========================================================================
# DERIVE: Firewall Logs
# Every session generates firewall logs (ALLOW/DENY based on state)
# ==========================================================================

def derive_firewall_action(state):
    """Derive firewall action from session state."""
    if state in ['BLOCKED']:
        return np.random.choice(['DENY', 'DROP', 'REJECT'], p=[0.5, 0.3, 0.2])
    return 'ALLOW'

# Get unique sessions (final state per session)
session_final = session_full.groupby('session_id').last().reset_index()

firewall_logs = session_final[['session_id', 'timestamp', 'src_ip', 'dest_ip', 
                                'src_port', 'dst_port', 'protocol', 'bytes_sent', 
                                'bytes_received', 'session_state']].copy()
firewall_logs['log_id'] = ['FW_' + str(i) for i in range(len(firewall_logs))]
firewall_logs['action'] = firewall_logs['session_state'].apply(derive_firewall_action)
firewall_logs['zone_src'] = 'internal'
firewall_logs['zone_dst'] = 'untrust'
firewall_logs['rule_id'] = np.random.choice(['RULE_001', 'RULE_002', 'RULE_003', 'RULE_010', 'RULE_DEFAULT'], 
                                            size=len(firewall_logs))

print(f"Firewall Logs: {len(firewall_logs)} records")
print("\nAction Distribution:")
print(firewall_logs['action'].value_counts())
firewall_logs.head(10)

Firewall Logs: 5000 records

Action Distribution:
action
ALLOW     4512
DENY       259
DROP       126
REJECT     103
Name: count, dtype: int64


,session_id,timestamp,src_ip,dest_ip,src_port,dst_port,protocol,bytes_sent,bytes_received,session_state,log_id,action,zone_src,zone_dst,rule_id
0,SESS_0,2025-02-01T00:06:00+00:00,10.16.231.61,13.198.56.18,55405,123,UDP,16611,257162,CLOSED,FW_0,ALLOW,internal,untrust,RULE_002
1,SESS_1,2025-02-01T00:02:00+00:00,10.1.26.51,199.128.186.65,57685,161,UDP,64237,268609,CLOSED,FW_1,ALLOW,internal,untrust,RULE_001
2,SESS_10,2025-02-01T00:08:00+00:00,172.16.154.194,64.32.187.13,50383,161,UDP,34594,30663,CLOSED,FW_2,ALLOW,internal,untrust,RULE_003
3,SESS_100,2025-02-01T00:06:00+00:00,172.3.117.61,72.56.9.222,52459,22,TCP,47284,11800,CLOSED,FW_3,ALLOW,internal,untrust,RULE_001
4,SESS_1000,2025-02-01T00:09:00+00:00,10.3.185.122,20.6.204.104,62539,3306,TCP,106675,41931,CLOSED,FW_4,ALLOW,internal,untrust,RULE_DEFAULT
5,SESS_1001,2025-02-01T00:08:00+00:00,10.1.186.109,208.188.119.216,55940,25,TCP,136213,174905,CLOSED,FW_5,ALLOW,internal,untrust,RULE_001
6,SESS_1002,2025-02-01T00:10:00+00:00,10.3.8.62,104.105.175.65,64777,389,TCP,2222,176142,CLOSED,FW_6,ALLOW,internal,untrust,RULE_DEFAULT
7,SESS_1003,2025-02-01T00:06:00+00:00,10.3.95.110,204.40.3.90,58046,3389,TCP,67573,108655,CLOSED,FW_7,ALLOW,internal,untrust,RULE_DEFAULT
8,SESS_1004,2025-02-01T00:06:00+00:00,10.3.215.105,151.109.90.244,57866,22,TCP,5268,423116,BLOCKED,FW_8,REJECT,internal,untrust,RULE_010
9,SESS_1005,2025-02-01T00:06:00+00:00,10.3.95.146,20.37.245.144,64560,23,TCP,42378,249914,CLOSED,FW_9,ALLOW,internal,untrust,RULE_002


In [19]:
# ==========================================================================
# DERIVE: IDS/IPS Alerts
# Only sessions that went through ALERTED state or have high risk scores
# ==========================================================================

# Filter for sessions that had alerts or high-risk destinations
alerted_sessions = session_full[
    (session_full['session_state'] == 'ALERTED') | 
    (session_full['is_suspicious'] == True) |
    (session_full['risk_score'] >= 50)
].copy()

# Get unique sessions
alerted_unique = alerted_sessions.groupby('session_id').first().reset_index()

ids_alerts = alerted_unique[['session_id', 'timestamp', 'src_ip', 'dest_ip', 
                             'dst_port', 'service_name', 'domain_name', 'risk_score']].copy()
ids_alerts['alert_id'] = ['IDS_' + str(i) for i in range(len(ids_alerts))]

# Assign signature IDs and names based on characteristics
signature_options = [
    ('SIG_1001', 'Potential C2 Communication', 'malware', 'high'),
    ('SIG_1002', 'Suspicious Outbound Connection', 'recon', 'medium'),
    ('SIG_1003', 'DNS Query to Known Bad Domain', 'malware', 'high'),
    ('SIG_1004', 'Unusual Port Activity', 'policy_violation', 'low'),
    ('SIG_1005', 'Large Data Transfer', 'exfiltration', 'medium'),
    ('SIG_1006', 'Lateral Movement Attempt', 'lateral_movement', 'critical'),
    ('SIG_1007', 'Brute Force Attempt', 'exploit', 'high'),
]

sig_choices = np.random.choice(len(signature_options), size=len(ids_alerts))
ids_alerts['signature_id'] = [signature_options[i][0] for i in sig_choices]
ids_alerts['signature_name'] = [signature_options[i][1] for i in sig_choices]
ids_alerts['category'] = [signature_options[i][2] for i in sig_choices]
ids_alerts['severity'] = [signature_options[i][3] for i in sig_choices]
ids_alerts['action'] = np.random.choice(['alert', 'alert', 'block'], size=len(ids_alerts))

print(f"IDS Alerts: {len(ids_alerts)} records")
print("\nSeverity Distribution:")
print(ids_alerts['severity'].value_counts())
print("\nCategory Distribution:")
print(ids_alerts['category'].value_counts())
ids_alerts.head(10)

IDS Alerts: 1501 records

Severity Distribution:
severity
high        641
medium      452
low         211
critical    197
Name: count, dtype: int64

Category Distribution:
category
malware             417
exfiltration        228
recon               224
exploit             224
policy_violation    211
lateral_movement    197
Name: count, dtype: int64


,session_id,timestamp,src_ip,dest_ip,dst_port,service_name,domain_name,risk_score,alert_id,signature_id,signature_name,category,severity,action
0,SESS_10,2025-02-01T00:00:00+00:00,172.16.154.194,64.32.187.13,161,SNMP,cloudflare.com,100,IDS_0,SIG_1006,Lateral Movement Attempt,lateral_movement,critical,alert
1,SESS_100,2025-02-01T00:00:00+00:00,172.3.117.61,72.56.9.222,22,SSH,youtube.com,80,IDS_1,SIG_1006,Lateral Movement Attempt,lateral_movement,critical,alert
2,SESS_1000,2025-02-01T00:00:00+00:00,10.3.185.122,20.6.204.104,3306,MYSQL,facebook.com,50,IDS_2,SIG_1002,Suspicious Outbound Connection,recon,medium,block
3,SESS_1001,2025-02-01T00:04:00+00:00,10.1.186.109,208.188.119.216,25,SMTP,linkedin.com,0,IDS_3,SIG_1006,Lateral Movement Attempt,lateral_movement,critical,alert
4,SESS_1003,2025-02-01T00:00:00+00:00,10.3.95.110,204.40.3.90,3389,RDP,slack.com,50,IDS_4,SIG_1001,Potential C2 Communication,malware,high,block
5,SESS_1004,2025-02-01T00:05:00+00:00,10.3.215.105,151.109.90.244,22,SSH,slack.com,0,IDS_5,SIG_1005,Large Data Transfer,exfiltration,medium,alert
6,SESS_1012,2025-02-01T00:00:00+00:00,10.1.26.244,17.187.85.14,445,SMB,youtube.com,100,IDS_6,SIG_1002,Suspicious Outbound Connection,recon,medium,alert
7,SESS_1016,2025-02-01T00:00:00+00:00,10.16.61.92,151.41.132.154,0,ICMP,google.com,50,IDS_7,SIG_1001,Potential C2 Communication,malware,high,alert
8,SESS_1020,2025-02-01T00:00:00+00:00,172.1.34.215,34.68.83.127,53,DNS,salesforce.com,100,IDS_8,SIG_1006,Lateral Movement Attempt,lateral_movement,critical,alert
9,SESS_1022,2025-02-01T00:00:00+00:00,172.1.34.215,34.68.83.127,53,DNS,salesforce.com,100,IDS_9,SIG_1005,Large Data Transfer,exfiltration,medium,block


In [20]:
# ==========================================================================
# DERIVE: DNS Logs
# Sessions that had DNS_LOOKUP state generate DNS logs
# ==========================================================================

dns_sessions = session_full[
    (session_full['action_type'] == 'dns_query') | 
    (session_full['action_type'] == 'dns_success') |
    (session_full['action_type'] == 'dns_failure')
].copy()

dns_logs = dns_sessions[['session_id', 'timestamp', 'src_ip', 'dest_ip', 
                         'domain_name', 'fk_src_host_id', 'action_type']].copy()
dns_logs['query_id'] = ['DNS_' + str(i) for i in range(len(dns_logs))]
dns_logs['query_name'] = dns_logs['domain_name']
dns_logs['query_type'] = np.random.choice(['A', 'A', 'A', 'AAAA', 'CNAME', 'MX', 'TXT'], size=len(dns_logs))
dns_logs['response_code'] = dns_logs['action_type'].map({
    'dns_query': 'NOERROR',
    'dns_success': 'NOERROR',
    'dns_failure': np.random.choice(['NXDOMAIN', 'SERVFAIL', 'REFUSED'])
}).fillna('NOERROR')
dns_logs['response_ip'] = dns_logs['dest_ip']
dns_logs['ttl'] = np.random.choice([300, 600, 900, 1800, 3600, 7200], size=len(dns_logs))
dns_logs['is_suspicious'] = dns_logs['domain_name'].str.contains('suspicious|malware|phishing', case=False, na=False)

print(f"DNS Logs: {len(dns_logs)} records")
print("\nResponse Code Distribution:")
print(dns_logs['response_code'].value_counts())
print("\nQuery Type Distribution:")
print(dns_logs['query_type'].value_counts())
dns_logs.head(10)

DNS Logs: 7064 records

Response Code Distribution:
response_code
NOERROR     6886
SERVFAIL     178
Name: count, dtype: int64

Query Type Distribution:
query_type
A        3079
AAAA     1001
TXT       999
CNAME     996
MX        989
Name: count, dtype: int64


,session_id,timestamp,src_ip,dest_ip,domain_name,fk_src_host_id,action_type,query_id,query_name,query_type,response_code,response_ip,ttl,is_suspicious
1,SESS_0,2025-02-01T00:01:00+00:00,10.16.231.61,13.198.56.18,malware-c2.net,HOST_170,dns_query,DNS_0,malware-c2.net,A,NOERROR,13.198.56.18,7200,True
2,SESS_0,2025-02-01T00:02:00+00:00,10.16.231.61,13.198.56.18,malware-c2.net,HOST_170,dns_success,DNS_1,malware-c2.net,TXT,NOERROR,13.198.56.18,3600,True
8,SESS_1,2025-02-01T00:01:00+00:00,10.1.26.51,199.128.186.65,malware-c2.net,HOST_127,dns_query,DNS_2,malware-c2.net,CNAME,NOERROR,199.128.186.65,1800,True
9,SESS_1,2025-02-01T00:02:00+00:00,10.1.26.51,199.128.186.65,malware-c2.net,HOST_127,dns_failure,DNS_3,malware-c2.net,A,SERVFAIL,199.128.186.65,7200,True
11,SESS_2,2025-02-01T00:01:00+00:00,10.16.133.225,199.91.5.90,api.internal-tool.com,HOST_102,dns_query,DNS_4,api.internal-tool.com,AAAA,NOERROR,199.91.5.90,900,False
12,SESS_2,2025-02-01T00:02:00+00:00,10.16.133.225,199.91.5.90,api.internal-tool.com,HOST_102,dns_success,DNS_5,api.internal-tool.com,MX,NOERROR,199.91.5.90,7200,False
18,SESS_3,2025-02-01T00:01:00+00:00,10.3.250.39,17.175.224.88,salesforce.com,HOST_53,dns_query,DNS_6,salesforce.com,A,NOERROR,17.175.224.88,300,False
19,SESS_3,2025-02-01T00:02:00+00:00,10.3.250.39,17.175.224.88,salesforce.com,HOST_53,dns_success,DNS_7,salesforce.com,AAAA,NOERROR,17.175.224.88,3600,False
46,SESS_6,2025-02-01T00:01:00+00:00,10.3.112.218,8.227.131.15,api.internal-tool.com,HOST_15,dns_query,DNS_8,api.internal-tool.com,MX,NOERROR,8.227.131.15,3600,False
47,SESS_6,2025-02-01T00:02:00+00:00,10.3.112.218,8.227.131.15,api.internal-tool.com,HOST_15,dns_success,DNS_9,api.internal-tool.com,TXT,NOERROR,8.227.131.15,3600,False


In [21]:
# ==========================================================================
# DERIVE: Proxy Logs
# HTTP/HTTPS sessions generate proxy logs
# ==========================================================================

http_sessions = session_full[
    session_full['service_name'].isin(['HTTP', 'HTTPS'])
].copy()

# Get unique sessions
http_unique = http_sessions.groupby('session_id').first().reset_index()

proxy_logs = http_unique[['session_id', 'timestamp', 'src_ip', 'fk_src_host_id',
                          'domain_name', 'bytes_sent', 'bytes_received', 'category']].copy()
proxy_logs['proxy_log_id'] = ['PROXY_' + str(i) for i in range(len(proxy_logs))]
proxy_logs['method'] = np.random.choice(['GET', 'GET', 'GET', 'POST', 'PUT', 'DELETE', 'CONNECT'], 
                                         size=len(proxy_logs))
proxy_logs['url'] = 'https://' + proxy_logs['domain_name'].fillna('unknown.com') + '/path/resource'
proxy_logs['user_agent'] = np.random.choice([
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Firefox/121.0',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 14_2) Safari/17.2',
    'curl/7.88.0',
    'python-requests/2.31.0',
], size=len(proxy_logs))
proxy_logs['response_code'] = np.random.choice([200, 200, 200, 200, 201, 301, 302, 400, 403, 404, 500], 
                                               size=len(proxy_logs))
proxy_logs['content_type'] = np.random.choice(['text/html', 'application/json', 'image/png', 
                                               'application/javascript', 'text/css'], size=len(proxy_logs))
proxy_logs['action'] = np.where(proxy_logs['category'] == 'suspicious', 'block', 'allow')

print(f"Proxy Logs: {len(proxy_logs)} records")
print("\nHTTP Method Distribution:")
print(proxy_logs['method'].value_counts())
print("\nResponse Code Distribution:")
print(proxy_logs['response_code'].value_counts())
proxy_logs.head(10)

Proxy Logs: 683 records

HTTP Method Distribution:
method
GET        279
PUT        116
DELETE     106
CONNECT     99
POST        83
Name: count, dtype: int64

Response Code Distribution:
response_code
200    254
500     78
404     64
302     63
301     61
201     57
400     54
403     52
Name: count, dtype: int64


,session_id,timestamp,src_ip,fk_src_host_id,domain_name,bytes_sent,bytes_received,category,proxy_log_id,method,url,user_agent,response_code,content_type,action
0,SESS_1010,2025-02-01T00:00:00+00:00,10.1.59.1,HOST_78,salesforce.com,199912,266376,social_media,PROXY_0,CONNECT,https://salesforce.com/path/resource,Mozilla/5.0 (Macintosh; Intel Mac OS X 14_2) S...,201,application/json,allow
1,SESS_102,2025-02-01T00:00:00+00:00,10.2.221.226,HOST_63,salesforce.com,1782,160017,uncategorized,PROXY_1,GET,https://salesforce.com/path/resource,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Fire...,500,text/html,allow
2,SESS_1041,2025-02-01T00:00:00+00:00,10.2.37.234,HOST_59,facebook.com,89128,140462,streaming,PROXY_2,PUT,https://facebook.com/path/resource,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chro...,200,image/png,allow
3,SESS_1052,2025-02-01T00:00:00+00:00,10.16.170.198,HOST_65,suspicious-domain.xyz,116946,49704,business,PROXY_3,DELETE,https://suspicious-domain.xyz/path/resource,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chro...,201,application/json,allow
4,SESS_1055,2025-02-01T00:00:00+00:00,10.2.156.231,HOST_64,suspicious-domain.xyz,51830,115180,business,PROXY_4,GET,https://suspicious-domain.xyz/path/resource,Mozilla/5.0 (Macintosh; Intel Mac OS X 14_2) S...,200,application/javascript,allow
5,SESS_1064,2025-02-01T00:00:00+00:00,10.2.107.72,HOST_58,facebook.com,6441,116462,streaming,PROXY_5,GET,https://facebook.com/path/resource,curl/7.88.0,200,image/png,allow
6,SESS_1083,2025-02-01T00:00:00+00:00,10.2.221.226,HOST_63,salesforce.com,28093,126775,uncategorized,PROXY_6,GET,https://salesforce.com/path/resource,Mozilla/5.0 (Macintosh; Intel Mac OS X 14_2) S...,200,application/json,allow
7,SESS_1084,2025-02-01T00:00:00+00:00,10.1.168.117,HOST_67,malware-c2.net,40591,80356,cloud_service,PROXY_7,GET,https://malware-c2.net/path/resource,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Fire...,400,text/html,allow
8,SESS_1086,2025-02-01T00:00:00+00:00,10.3.215.86,HOST_73,microsoft.com,84674,188969,cdn,PROXY_8,DELETE,https://microsoft.com/path/resource,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chro...,301,application/javascript,allow
9,SESS_1088,2025-02-01T00:00:00+00:00,172.1.190.214,HOST_75,linkedin.com,15105,40366,business,PROXY_9,PUT,https://linkedin.com/path/resource,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Fire...,200,image/png,allow


In [22]:
# ==========================================================================
# DERIVE: Netflow Records
# Every session generates netflow records (router-level flow data)
# ==========================================================================

netflow_records = session_final[['session_id', 'timestamp', 'src_ip', 'dest_ip',
                                  'src_port', 'dst_port', 'protocol', 'bytes_sent',
                                  'bytes_received', 'packets', 'duration_ms']].copy()
netflow_records['flow_id'] = ['FLOW_' + str(i) for i in range(len(netflow_records))]
netflow_records['router_id'] = np.random.choice(['RTR_CORE_01', 'RTR_CORE_02', 'RTR_EDGE_01'], 
                                                 size=len(netflow_records))
netflow_records['input_interface'] = np.random.choice(['Gi0/0', 'Gi0/1', 'Gi0/2', 'Gi1/0', 'Gi1/1'], 
                                                      size=len(netflow_records))
netflow_records['output_interface'] = np.random.choice(['Gi0/0', 'Gi0/1', 'Gi0/2', 'Gi1/0', 'Gi1/1'], 
                                                       size=len(netflow_records))
netflow_records['tcp_flags'] = np.random.choice(['SYN', 'ACK', 'PSH-ACK', 'FIN-ACK', 'RST'], 
                                                 size=len(netflow_records))
netflow_records['tos'] = np.random.choice([0, 0, 0, 32, 40, 46], size=len(netflow_records))
netflow_records['total_bytes'] = netflow_records['bytes_sent'] + netflow_records['bytes_received']

print(f"Netflow Records: {len(netflow_records)} records")
print("\nRouter Distribution:")
print(netflow_records['router_id'].value_counts())
print("\nProtocol Distribution:")
print(netflow_records['protocol'].value_counts())
netflow_records.head(10)

Netflow Records: 5000 records

Router Distribution:
router_id
RTR_CORE_02    1732
RTR_EDGE_01    1662
RTR_CORE_01    1606
Name: count, dtype: int64

Protocol Distribution:
protocol
TCP     3647
UDP     1026
ICMP     327
Name: count, dtype: int64


,session_id,timestamp,src_ip,dest_ip,src_port,dst_port,protocol,bytes_sent,bytes_received,packets,duration_ms,flow_id,router_id,input_interface,output_interface,tcp_flags,tos,total_bytes
0,SESS_0,2025-02-01T00:06:00+00:00,10.16.231.61,13.198.56.18,55405,123,UDP,16611,257162,33,31720,FLOW_0,RTR_CORE_02,Gi1/1,Gi1/1,SYN,40,273773
1,SESS_1,2025-02-01T00:02:00+00:00,10.1.26.51,199.128.186.65,57685,161,UDP,64237,268609,70,71566,FLOW_1,RTR_CORE_02,Gi1/1,Gi0/0,PSH-ACK,40,332846
2,SESS_10,2025-02-01T00:08:00+00:00,172.16.154.194,64.32.187.13,50383,161,UDP,34594,30663,73,151358,FLOW_2,RTR_CORE_01,Gi0/2,Gi1/1,RST,46,65257
3,SESS_100,2025-02-01T00:06:00+00:00,172.3.117.61,72.56.9.222,52459,22,TCP,47284,11800,31,15335,FLOW_3,RTR_EDGE_01,Gi1/1,Gi0/1,ACK,0,59084
4,SESS_1000,2025-02-01T00:09:00+00:00,10.3.185.122,20.6.204.104,62539,3306,TCP,106675,41931,4,5161,FLOW_4,RTR_EDGE_01,Gi1/0,Gi0/0,FIN-ACK,0,148606
5,SESS_1001,2025-02-01T00:08:00+00:00,10.1.186.109,208.188.119.216,55940,25,TCP,136213,174905,136,2527,FLOW_5,RTR_EDGE_01,Gi1/1,Gi1/1,ACK,0,311118
6,SESS_1002,2025-02-01T00:10:00+00:00,10.3.8.62,104.105.175.65,64777,389,TCP,2222,176142,52,9341,FLOW_6,RTR_CORE_02,Gi0/0,Gi0/2,RST,46,178364
7,SESS_1003,2025-02-01T00:06:00+00:00,10.3.95.110,204.40.3.90,58046,3389,TCP,67573,108655,68,4054,FLOW_7,RTR_EDGE_01,Gi1/1,Gi1/1,FIN-ACK,40,176228
8,SESS_1004,2025-02-01T00:06:00+00:00,10.3.215.105,151.109.90.244,57866,22,TCP,5268,423116,68,4329,FLOW_8,RTR_CORE_02,Gi0/2,Gi0/2,RST,40,428384
9,SESS_1005,2025-02-01T00:06:00+00:00,10.3.95.146,20.37.245.144,64560,23,TCP,42378,249914,271,4817,FLOW_9,RTR_CORE_01,Gi1/1,Gi1/0,ACK,0,292292


## Validate Cross-Entity Correlation

In [23]:
print("="*60)
print("CROSS-ENTITY CORRELATION VALIDATION")
print("="*60)

# 1. User-Host relationship
user_host_valid = host_df['fk_primary_user_id'].isin(user_df['user_id']).all()
print(f"\n1. User-Host FK validity: {user_host_valid}")
print(f"   Hosts with assigned users: {host_df['fk_primary_user_id'].notna().sum()}/{len(host_df)}")

# 2. Session-Host relationship
session_host_valid = session_df['fk_src_host_id'].isin(host_df['host_id']).all()
print(f"\n2. Session-Host FK validity: {session_host_valid}")

# 3. Firewall logs reference valid sessions
fw_session_valid = firewall_logs['session_id'].isin(session_df['session_id']).all()
print(f"\n3. Firewall-Session FK validity: {fw_session_valid}")

# 4. IDS alerts reference valid sessions
ids_session_valid = ids_alerts['session_id'].isin(session_df['session_id']).all()
print(f"\n4. IDS-Session FK validity: {ids_session_valid}")

# 5. DNS logs reference valid sessions
dns_session_valid = dns_logs['session_id'].isin(session_df['session_id']).all()
print(f"\n5. DNS-Session FK validity: {dns_session_valid}")

# 6. Proxy logs reference valid sessions
proxy_session_valid = proxy_logs['session_id'].isin(session_df['session_id']).all()
print(f"\n6. Proxy-Session FK validity: {proxy_session_valid}")

# 7. Netflow records reference valid sessions
netflow_session_valid = netflow_records['session_id'].isin(session_df['session_id']).all()
print(f"\n7. Netflow-Session FK validity: {netflow_session_valid}")

# 8. VPN events reference valid users
vpn_user_valid = vpn_df['fk_user_id'].isin(user_df['user_id']).all()
print(f"\n8. VPN-User FK validity: {vpn_user_valid}")

# 9. Auth events reference valid users and hosts
auth_user_valid = auth_df['fk_user_id'].isin(user_df['user_id']).all()
auth_host_valid = auth_df['fk_host_id'].isin(host_df['host_id']).all()
print(f"\n9. Auth-User FK validity: {auth_user_valid}")
print(f"   Auth-Host FK validity: {auth_host_valid}")

# 10. Endpoint events reference valid hosts
endpoint_host_valid = endpoint_df['fk_host_id'].isin(host_df['host_id']).all()
print(f"\n10. Endpoint-Host FK validity: {endpoint_host_valid}")

CROSS-ENTITY CORRELATION VALIDATION

1. User-Host FK validity: True
   Hosts with assigned users: 200/200

2. Session-Host FK validity: True

3. Firewall-Session FK validity: True

4. IDS-Session FK validity: True

5. DNS-Session FK validity: True

6. Proxy-Session FK validity: True

7. Netflow-Session FK validity: True

8. VPN-User FK validity: True

9. Auth-User FK validity: True
   Auth-Host FK validity: True

10. Endpoint-Host FK validity: True


In [24]:
print("="*60)
print("CORRELATION STATISTICS")
print("="*60)

# Session to derived log ratios
unique_sessions = session_df['session_id'].nunique()
print(f"\nUnique Network Sessions: {unique_sessions}")
print(f"  -> Firewall Logs: {len(firewall_logs)} ({len(firewall_logs)/unique_sessions:.1f}x)")
print(f"  -> IDS Alerts: {len(ids_alerts)} ({100*len(ids_alerts)/unique_sessions:.1f}% of sessions)")
print(f"  -> DNS Logs: {len(dns_logs)} ({100*len(dns_logs)/unique_sessions:.1f}% of sessions)")
print(f"  -> Proxy Logs: {len(proxy_logs)} ({100*len(proxy_logs)/unique_sessions:.1f}% of sessions)")
print(f"  -> Netflow Records: {len(netflow_records)} ({len(netflow_records)/unique_sessions:.1f}x)")

# Session state distribution impact
session_states = session_final['session_state'].value_counts()
print(f"\nSession Final States:")
for state, count in session_states.items():
    print(f"  {state}: {count} ({100*count/unique_sessions:.1f}%)")

# Blocked sessions -> should correlate with FW DENY actions
blocked_sessions = len(session_final[session_final['session_state'] == 'BLOCKED'])
fw_denied = len(firewall_logs[firewall_logs['action'] != 'ALLOW'])
print(f"\nBlocked Sessions: {blocked_sessions}")
print(f"Firewall Denied/Dropped: {fw_denied}")

CORRELATION STATISTICS

Unique Network Sessions: 5000
  -> Firewall Logs: 5000 (1.0x)
  -> IDS Alerts: 1501 (30.0% of sessions)
  -> DNS Logs: 7064 (141.3% of sessions)
  -> Proxy Logs: 683 (13.7% of sessions)
  -> Netflow Records: 5000 (1.0x)

Session Final States:
  CLOSED: 4512 (90.2%)
  BLOCKED: 488 (9.8%)

Blocked Sessions: 488
Firewall Denied/Dropped: 488


## Save All Data to CSV

In [25]:
import os

# Create output directory
output_dir = "siem_edr_data"
os.makedirs(output_dir, exist_ok=True)

# Save core entities
user_df.to_csv(f"{output_dir}/users.csv", index=False)
host_df.to_csv(f"{output_dir}/hosts.csv", index=False)
ext_dest_df.to_csv(f"{output_dir}/external_destinations.csv", index=False)
service_df.to_csv(f"{output_dir}/service_definitions.csv", index=False)

# Save generated entities
session_df.to_csv(f"{output_dir}/network_sessions.csv", index=False)
endpoint_df.to_csv(f"{output_dir}/endpoint_events.csv", index=False)
vpn_df.to_csv(f"{output_dir}/vpn_events.csv", index=False)
auth_df.to_csv(f"{output_dir}/auth_events.csv", index=False)

# Save derived logs
firewall_logs.to_csv(f"{output_dir}/firewall_logs.csv", index=False)
ids_alerts.to_csv(f"{output_dir}/ids_alerts.csv", index=False)
dns_logs.to_csv(f"{output_dir}/dns_logs.csv", index=False)
proxy_logs.to_csv(f"{output_dir}/proxy_logs.csv", index=False)
netflow_records.to_csv(f"{output_dir}/netflow_records.csv", index=False)

# Save joined session data for analysis
session_full.to_csv(f"{output_dir}/network_sessions_full.csv", index=False)

print(f"Data saved to '{output_dir}/' directory:")
for f in sorted(os.listdir(output_dir)):
    size = os.path.getsize(f"{output_dir}/{f}")
    print(f"  - {f}: {size/1024:.1f} KB")

Data saved to 'siem_edr_data/' directory:
  - auth_events.csv: 1659.3 KB
  - dns_logs.csv: 1068.3 KB
  - endpoint_events.csv: 4543.6 KB
  - external_destinations.csv: 6.2 KB
  - firewall_logs.csv: 663.1 KB
  - hosts.csv: 14.7 KB
  - ids_alerts.csv: 224.2 KB
  - netflow_records.csv: 711.3 KB
  - network_sessions.csv: 4174.8 KB
  - network_sessions_full.csv: 7996.5 KB
  - proxy_logs.csv: 140.6 KB
  - service_definitions.csv: 0.3 KB
  - users.csv: 5.7 KB
  - vpn_events.csv: 350.6 KB


## Summary

This notebook generated synthetic SIEM/EDR data with the following characteristics:

### Entities Generated

| Entity | Description | Records |
|--------|-------------|--------:|
| Users | System users | 150 |
| Hosts | Network endpoints | 200 |
| External Destinations | External IPs/domains | 100 |
| Service Definitions | Protocol/port mappings | 15 |
| Network Sessions | Stateful network connections | ~5,000+ |
| Endpoint Events | EDR process/file events | 10,000 |
| VPN Events | Remote access events | 500 |
| Auth Events | Authentication logs | 3,000 |

### Derived Logs (Correlated)

| Log Type | Correlation Source | Records |
|----------|-------------------|--------:|
| Firewall Logs | All sessions | ~5,000 |
| IDS Alerts | Suspicious sessions | ~250-500 |
| DNS Logs | DNS lookup sessions | ~3,500 |
| Proxy Logs | HTTP/HTTPS sessions | ~2,000 |
| Netflow Records | All sessions | ~5,000 |

### Key Features

1. **Stateful Network Sessions**: Sessions follow realistic state machine transitions (INIT → DNS → CONNECT → ESTABLISHED → DATA → CLOSE)

2. **Cross-Entity Correlation**: 
   - Users → Hosts (primary assignment)
   - Hosts → Sessions (source of network activity)
   - Sessions → All network logs (firewall, IDS, DNS, proxy, netflow)
   - Users → VPN/Auth events

3. **Realistic Distributions**:
   - ~5% of sessions blocked by firewall
   - ~5% of sessions trigger IDS alerts
   - ~70% of sessions require DNS lookup
   - ~40% HTTP/HTTPS traffic through proxy

4. **Security-Relevant Data**:
   - Suspicious domains flagged
   - Risk scores for destinations
   - Auth failures and anomalies
   - Endpoint process trees with severity

### Potential Extensions

- Add attack scenario injection (brute force, lateral movement, exfiltration)
- Include DHCP/ARP logs
- Add email gateway logs
- Include cloud service (AWS/Azure/GCP) audit logs
- Add time-based patterns (business hours vs off-hours activity)